In [1]:
import pandas as pd
from collections import defaultdict
from datetime import datetime
import pyarrow as pa
import pyarrow.parquet as pq
import os

base_path = "../data/vitals_data/mimic_vitals_p0{}.parquet"

patient_records = defaultdict(set)

for i in range(10):  # p00 → p09
    path = base_path.format(i)
    print(f"Processing {path}...")

    df = pd.read_parquet(path, columns=["Patient", "Record"])

    for patient, group in df.groupby("Patient"):
        patient_records[patient].update(group["Record"].unique())

# convert sets to lists if needed
patient_records = {k: list(v) for k, v in patient_records.items()}

def parse_record_start(record_name: str) -> datetime:
    parts = record_name.split("-")
    year, month, day = parts[1], parts[2], parts[3]
    hour = parts[4]
    minute = parts[5].replace("n", "").replace("p", "")  # strip suffix
    return datetime(int(year), int(month), int(day), int(hour), int(minute))

record_starts = {
    record: parse_record_start(record)
    for records in patient_records.values()
    for record in records
}

subject_record_map: dict[int, list[tuple[str, datetime]]] = {}
for key, records in patient_records.items():
    subj_id = int(key[1:])  # 'p000079' -> 79
    subject_record_map[subj_id] = [(r, record_starts[r]) for r in records]

valid_subject_ids = set(subject_record_map.keys())

Processing ../data/vitals_data/mimic_vitals_p00.parquet...
Processing ../data/vitals_data/mimic_vitals_p01.parquet...
Processing ../data/vitals_data/mimic_vitals_p02.parquet...
Processing ../data/vitals_data/mimic_vitals_p03.parquet...
Processing ../data/vitals_data/mimic_vitals_p04.parquet...
Processing ../data/vitals_data/mimic_vitals_p05.parquet...
Processing ../data/vitals_data/mimic_vitals_p06.parquet...
Processing ../data/vitals_data/mimic_vitals_p07.parquet...
Processing ../data/vitals_data/mimic_vitals_p08.parquet...
Processing ../data/vitals_data/mimic_vitals_p09.parquet...


In [ ]:
DATA_DIR = "../data/clinical"
INPUT_FILE = os.path.join(DATA_DIR, "CHARTEVENTS.csv")
OUTPUT_FILE = os.path.join(DATA_DIR, "CLEANED_CHARTEVENTS.parquet")
D_ITEMS_FILE = os.path.join(DATA_DIR, "D_ITEMS.csv")

CHUNK_SIZE = 10_000_000

KEEP_CATEGORIES = {
    # Labs
    "Labs", "ABG", "ABG'S", "ABG's", "Other ABGs",
    "VBG'S", "VBG's", "Venous ABG",
    "Mixed Venous Gases", "Blood Gases",
    "Hematology", "Heme/Coag", "Coags",
    "Chemistry",
    "Enzymes",
    "Drug Level", "Drug level",
    "Toxicology",
    "CSF",
    "Urine",

    # Monitoring
    "Hemodynamics",
    "Neurological",
    "Respiratory", "2-Ventilation", "Pulmonary",
    "Cardiovascular", "Cardiovascular (Pulses)",
    "Pain/Sedation",
    "GI/GU",

    # Severity scores — strong static/low-freq conditioning
    "Scores - APACHE II", "Scores - APACHE IV", "Scores - APACHE IV (2)",
    "ApacheII Parameters", "ApacheIV Parameters",

    # Nutrition/fluids — relevant for hemodynamic state
    "Nutrition - Enteral", "Nutrition - Parenteral", "Nutrition - Supplements",
    "Fluids/Intake", "Blood Products/Colloids",

    # Interventions worth conditioning on
    "Dialysis",
    "Output",
    "Antibiotics", "ANTIBACTERIUM",
}

USECOLS = ["SUBJECT_ID", "ITEMID", "CHARTTIME", "VALUE", "VALUEUOM"]


def load_d_items(path: str) -> pd.DataFrame:
    items = pd.read_csv(path, usecols=["ITEMID", "LABEL", "CATEGORY"]).set_index("ITEMID")
    filtered = items[items["CATEGORY"].isin(KEEP_CATEGORIES)]
    return filtered

def get_phrase(df):
    df["PHRASE"] = (
        df["LABEL"].str.strip()
        + " "
        + df["VALUE"].astype(str)
        + " "
        + df["VALUEUOM"].str.strip().fillna("")
    ).str.strip()
    return df

def parse_record_start(record_name: str) -> datetime:
    parts = record_name.split("-")
    year, month, day = parts[1], parts[2], parts[3]
    hour = parts[4]
    minute = parts[5].replace("n", "").replace("p", "")  # strip suffix
    return datetime(int(year), int(month), int(day), int(hour), int(minute))


def subject_id_to_key(subject_id: int) -> str:
    # 79 -> 'p000079'
    return f"p{subject_id:06d}"

def get_relative_time(df):
    df["CHARTTIME"] = pd.to_datetime(df["CHARTTIME"])
        
    records_df = pd.DataFrame([
        {"SUBJECT_ID": subj_id, "RECORD": record, "START_TIME": start_time}
        for subj_id, pairs in subject_record_map.items()
        for record, start_time in pairs
    ])
    records_df["SUBJECT_ID"] = records_df["SUBJECT_ID"].astype("int32")

    expanded = df.merge(records_df, on="SUBJECT_ID", how="inner")

    expanded["MINUTE"] = (
        (expanded["CHARTTIME"] - expanded["START_TIME"])
        .dt.total_seconds()
        .div(60)
        .round()
        .astype("int32")
    )

    result = (
        expanded
        .drop(columns=["START_TIME"])
        .sort_values(["RECORD", "MINUTE"])
        .reset_index(drop=True)
    )

    return result


def run_chartevents_etl():
    d_items = load_d_items(D_ITEMS_FILE)

    reader = pd.read_csv(
        INPUT_FILE,
        usecols=USECOLS,
        chunksize=CHUNK_SIZE,
        low_memory=False,
        dtype={"SUBJECT_ID": "int32", "ITEMID": "int32", "VALUEUOM": "str", "VALUE": "str"},
    )

    writer = None

    for i, chunk in enumerate(reader):
        print(f"Processing chunk {i + 1} ({len(chunk):,} rows)...")

        processed = (
            chunk
            .join(d_items, on="ITEMID", how="inner")
            .reset_index(drop=True)
        )

        processed = get_phrase(processed)
        print("  -> Phrases generated")

        result = get_relative_time(processed)
        print("  -> Relative timestamp calculated")

        result = result[["RECORD", "MINUTE", "PHRASE"]].rename(columns={"RECORD": "Record", "MINUTE": "Minute", "PHRASE": "Phrase"})

        table = pa.Table.from_pandas(result, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(OUTPUT_FILE, table.schema)
        writer.write_table(table)

        print(f"  -> {len(result):,} rows written.")

    if writer:
        writer.close()

    print(f"\nDone. Output written to {OUTPUT_FILE}")


In [3]:
run_chartevents_etl()

Processing chunk 1 (10,000,000 rows)...
  -> Phrases generated
  -> Relative timestamp calculated
  -> 6,087,478 rows written.
Processing chunk 2 (10,000,000 rows)...
  -> Phrases generated
  -> Relative timestamp calculated
  -> 5,301,612 rows written.
Processing chunk 3 (10,000,000 rows)...
  -> Phrases generated
  -> Relative timestamp calculated
  -> 6,206,359 rows written.
Processing chunk 4 (10,000,000 rows)...
  -> Phrases generated
  -> Relative timestamp calculated
  -> 2,166,637 rows written.
Processing chunk 5 (10,000,000 rows)...
  -> Phrases generated
  -> Relative timestamp calculated
  -> 265,216 rows written.
Processing chunk 6 (10,000,000 rows)...
  -> Phrases generated
  -> Relative timestamp calculated
  -> 416,228 rows written.
Processing chunk 7 (10,000,000 rows)...
  -> Phrases generated
  -> Relative timestamp calculated
  -> 321,301 rows written.
Processing chunk 8 (10,000,000 rows)...
  -> Phrases generated
  -> Relative timestamp calculated
  -> 267,513 rows w

In [12]:
MV_FILE = os.path.join(DATA_DIR, "INPUTEVENTS_MV.csv")
CV_FILE = os.path.join(DATA_DIR, "INPUTEVENTS_CV.csv")
OUTPUT_FILE = os.path.join(DATA_DIR, "CLEANED_INPUTEVENTS.parquet")

def run_inputevents_etl():
    d_items = pd.read_csv(D_ITEMS_FILE, usecols=["ITEMID", "LABEL", "CATEGORY"]).set_index("ITEMID")

    mv_events = pd.read_csv(MV_FILE, usecols=["SUBJECT_ID", "ITEMID", "STARTTIME", "AMOUNT", "AMOUNTUOM"]).rename(columns={"STARTTIME": "CHARTTIME"})
    print("MV Events read")
    cv_events = pd.read_csv(CV_FILE, usecols=["SUBJECT_ID", "ITEMID", "CHARTTIME", "AMOUNT", "AMOUNTUOM"], dtype={
        "SUBJECT_ID": "int32",
        "ITEMID": "int32",
        "AMOUNT": "float32",
        "AMOUNTUOM": "str",
        "CHARTTIME": "str",
    })
    print("CV Events read")

    processed = pd.concat([mv_events, cv_events]).rename(columns={"AMOUNT": "VALUE", "AMOUNTUOM": "VALUEUOM"})
    processed = (
        processed
        .join(d_items, on="ITEMID", how="inner")
        .reset_index(drop=True)
    )
    print("Item info joined")

    processed = get_phrase(processed)
    print("  -> Phrases generated")

    result = get_relative_time(processed)
    print("  -> Relative timestamp calculated")

    result = result[["RECORD", "MINUTE", "PHRASE"]].rename(columns={"RECORD": "Record", "MINUTE": "Minute", "PHRASE": "Phrase"})

    table = pa.Table.from_pandas(result, preserve_index=False)
    writer = pq.ParquetWriter(OUTPUT_FILE, table.schema)
    writer.write_table(table)

    print(f"  -> {len(result):,} rows written.")

    if writer:
        writer.close()

    print(f"\nDone. Output written to {OUTPUT_FILE}")

In [13]:
run_inputevents_etl()

MV Events read
CV Events read
Item info joined
  -> Phrases generated
  -> Relative timestamp calculated
  -> 11,577,618 rows written.

Done. Output written to ../data/clinical\CLEANED_INPUTEVENTS.parquet


In [16]:
pd.read_parquet("../data/clinical/CLEANED_CHARTEVENTS.parquet")

,Record,Minute,Phrase
0,p000107-2121-11-30-20-03n,-16,Potassium (whole blood) 6 mEq/L
1,p000107-2121-11-30-20-03n,3,Respiratory Rate 19 insp/min
2,p000107-2121-11-30-20-03n,5,O2 saturation pulseoxymetry 97 %
3,p000107-2121-11-30-20-03n,7,Respiratory Rate 18 insp/min
4,p000107-2121-11-30-20-03n,7,O2 saturation pulseoxymetry 99 %
...,...,...,...
53490082,p099999-2117-12-31-19-07n,1044,Cough Effort Strong
53490083,p099999-2117-12-31-19-07n,1044,Cough Type Non-productive
53490084,p099999-2117-12-31-19-07n,1044,Capillary Refill L Normal <3 Seconds
53490085,p099999-2117-12-31-19-07n,1044,Response Localizes


In [15]:
pd.read_parquet("../data/clinical/CLEANED_INPUTEVENTS.parquet")

,Record,Minute,Phrase
0,p000020-2183-04-28-17-47n,13,OR Crystalloid 3100.0 ml
1,p000020-2183-04-28-17-47n,13,OR Colloid 375.0 ml
2,p000020-2183-04-28-17-47n,13,D5W 0.0 ml
3,p000020-2183-04-28-17-47n,13,NaN
4,p000020-2183-04-28-17-47n,13,D5W 0.0 ml
...,...,...,...
11577613,p099999-2117-12-31-19-07n,293,LR 669.3333668 ml
11577614,p099999-2117-12-31-19-07n,773,Heparin Sodium (Prophylaxis) 1.0 dose
11577615,p099999-2117-12-31-19-07n,813,Piggyback 49.9999986 ml
11577616,p099999-2117-12-31-19-07n,813,Magnesium Sulfate 2.0000001 grams
